In [ ]:

import pandas as pd

# 定义训练数据和测试数据的路径
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/media_campaign_cost/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/media_campaign_cost/test.csv'

# 加载训练数据和测试数据
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 查看训练数据的前几行
train_df.head()


Displayed on the user's machine.

In [ ]:

# 查看训练数据的基本信息和统计信息
train_df.info()
train_df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28827 entries, 0 to 28826
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    28827 non-null  int64  
 1   store_sales           28827 non-null  float64
 2   unit_sales            28827 non-null  int64  
 3   total_children        28827 non-null  int64  
 4   num_children_at_home  28827 non-null  int64  
 5   avg_cars_at_home      28827 non-null  int64  
 6   gross_weight          28827 non-null  float64
 7   recyclable_package    28827 non-null  int64  
 8   low_fat               28827 non-null  int64  
 9   units_per_case        28827 non-null  int64  
 10  store_sqft            28827 non-null  int64  
 11  coffee_bar            28827 non-null  int64  
 12  video_store           28827 non-null  int64  
 13  salad_bar             28827 non-null  int64  
 14  prepared_food         28827 non-null  int64  
 15  florist            

In [ ]:

# 查看训练数据的基本信息和统计信息
train_df.info()
train_df.describe()


Displayed on the user's machine.

In [ ]:


# 查看测试数据的基本信息和统计信息
test_df.info()
test_df.describe()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7207 entries, 0 to 7206
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    7207 non-null   int64  
 1   store_sales           7207 non-null   float64
 2   unit_sales            7207 non-null   int64  
 3   total_children        7207 non-null   int64  
 4   num_children_at_home  7207 non-null   int64  
 5   avg_cars_at_home      7207 non-null   int64  
 6   gross_weight          7207 non-null   float64
 7   recyclable_package    7207 non-null   int64  
 8   low_fat               7207 non-null   int64  
 9   units_per_case        7207 non-null   int64  
 10  store_sqft            7207 non-null   int64  
 11  coffee_bar            7207 non-null   int64  
 12  video_store           7207 non-null   int64  
 13  salad_bar             7207 non-null   int64  
 14  prepared_food         7207 non-null   int64  
 15  florist              

In [ ]:


# 查看测试数据的基本信息和统计信息
test_df.info()
test_df.describe()



Displayed on the user's machine.

In [ ]:


# 数据预处理和特征工程

# 选择特征和目标变量
features = ['store_sales', 'unit_sales', 'total_children', 'num_children_at_home', 'avg_cars_at_home', 
            'gross_weight', 'recyclable_package', 'low_fat', 'units_per_case', 'store_sqft', 
            'coffee_bar', 'video_store', 'salad_bar', 'prepared_food', 'florist']
target = 'cost'

# 分离特征和目标变量
X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# 数据标准化
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 对目标变量进行对数转换
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 查看标准化后的特征
pd.DataFrame(X_train_scaled, columns=features).head()



---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[5], line 42
     40 pass
     41 print('##active_line23##')
---> 42 y_train_log = np.log1p(y_train)
     43 print('##active_line24##')
     44 y_test_log = np.log1p(y_test)

NameError: name 'np' is not defined

In [ ]:

import numpy as np

# 重新进行对数转换
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 查看标准化后的特征
pd.DataFrame(X_train_scaled, columns=features).head()


Displayed on the user's machine.

In [ ]:



# 选择并训练模型

# 导入模型
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_log_error

# 定义模型
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

# 训练和评估模型
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train_log)
    y_pred_log = model.predict(X_test_scaled)
    y_pred = np.expm1(y_pred_log)
    rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))
    results[name] = rmsle

# 显示结果
results


{'Linear Regression': np.float64(0.3169135758242533),
 'Decision Tree': np.float64(0.43892761092367993),
 'Random Forest': np.float64(0.31536303134785265),
 'Gradient Boosting': np.float64(0.3101190738190712)}

In [ ]:


# 使用梯度提升模型进行预测
best_model = GradientBoostingRegressor(random_state=42)
best_model.fit(X_train_scaled, y_train_log)
y_pred_log = best_model.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)

# 创建提交文件
submission = pd.DataFrame({
    'id': test_df['id'],
    'cost': y_pred
})

# 保存提交文件
submission.to_csv('submission.csv', index=False)

# 查看前几行提交文件
submission.head()



Displayed on the user's machine.

In [ ]:


# 使用梯度提升模型进行预测
best_model = GradientBoostingRegressor(random_state=42)
best_model.fit(X_train_scaled, y_train_log)
y_pred_log = best_model.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)

# 创建提交文件
submission = pd.DataFrame({
    'id': test_df['id'],
    'cost': y_pred
})

# 保存提交文件
submission.to_csv('submission.csv', index=False)

# 查看前几行提交文件
submission.head()



`csv` disabled or not supported.